# Vanilla Policy Gradient

In [ ]:
import gymnasium as gym
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.init import xavier_uniform_
import numpy as np

GAMMA = 0.99
LEARNING_RATE = 0.01
BATCH_SIZE = 4
DEVICE = torch.device('mps')


class XavierLinear(nn.Linear):
    def __init__(self, in_features: int, out_features: int, bias: bool = True, device=None, dtype=None) -> None:
        super().__init__(in_features, out_features, bias, device, dtype)
        xavier_uniform_(self.weight)


class VPG(nn.Module):
    def __init__(self, input_size, output_size):
        super(VPG, self).__init__()
        self.net = nn.Sequential(
            XavierLinear(input_size, 128),
            nn.ReLU(),
            XavierLinear(128, output_size), 
        )

    def forward(self, x):
        return F.softmax(self.net(x), dim=0)

In [ ]:
env = gym.make("CartPole-v1")
model = VPG(env.observation_space.shape[0],
            2).to(DEVICE)
            
def run_episode(model, env, render=False):
    obs = env.reset()[0]
    if render:
        env.render()
    obs = torch.Tensor(env.reset()[0]).to(DEVICE)
    te = tr = False
    rewards, outputs, actions = [], [], []
    while not (te or tr):
        probs = model(obs)
        action = probs.multinomial(1).item()
        obs, r, te, tr, _ = env.step(action)
        obs = torch.Tensor(obs).to(DEVICE)
        r = -1 if te else 1 if tr else 0
        rewards.append(r)
        outputs.append(probs)
        actions.append(action)
    return torch.Tensor(rewards).to(DEVICE), torch.concatenate(outputs).reshape(len(rewards), 2), actions

### Discounted reward
For every step $t$ of every episode $k$, calculate discounted reward:

$ Q_{k,t} = \sum_{i=0}{\gamma^{i-t} r_i} $

In [ ]:
# rewards = torch.ones(8)

def discount_rewards(rewards):
    discounted_r = torch.zeros_like(rewards)
    additive_r = 0
    for idx in range(len(rewards)-1, -1, -1):
        to_add = GAMMA * additive_r
        additive_r = to_add + rewards[idx]
        discounted_r[idx] = additive_r
    return discounted_r.to(DEVICE)

### Training loop
The basic loss function:
$ L = -\sum_{k,t}Q_{k,t}log\pi_{\theta}(a_t | s_t)$

In the following example, we're using a slightly modified version. We're removing the baseline to lower the variance in the rewards. 

$ L = -\sum_{k,t}log\pi_{\theta}(a_t | s_t)(Q_{k,t} - b_k) $

Where $ b = \frac{1}{T} \sum_{t}^TQ_t $


In [ ]:
env = gym.make("CartPole-v1")
model = VPG(env.observation_space.shape[0],
            2).to(DEVICE)
def loss_function(discounted_r, probs, actions):
    logprobs = torch.log(probs)
    selected = logprobs[range(probs.shape[0]), actions]
    discounted_r = (discounted_r - discounted_r.mean()) / discounted_r.std() # This acts as removing the baseline. It's the simplest approach.
    weighted = selected * discounted_r
    return - weighted.mean()

rewards, probs, actions = run_episode(model, env)
disc = discount_rewards(rewards)
loss = loss_function(disc, probs, actions)
disc, (disc - disc.mean()) / disc.std()

In [ ]:
episode_total_reward = 0
batch_losses = torch.Tensor().to(DEVICE)
batch_actions = []
batch_disc_r = torch.Tensor().to(DEVICE)
batch_probs = torch.Tensor().to(DEVICE)
entropies = []
best_ep_reward = 0
losses, ep_total_lenghts = [], [0]

episodes = 0
TARGET_REWARD = 295

env = gym.make("CartPole-v1")
model = VPG(env.observation_space.shape[0],
            2).to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
torch.save(model.state_dict(), "results/model_0_eps.pt")

while np.mean(ep_total_lenghts[-100:]) < TARGET_REWARD:
    rewards, probs, actions = run_episode(model, env)
    episode_entropy = -(probs * probs.log()).sum(dim=1).mean()
    entropies.append(episode_entropy.detach().item())
    discounted_r = discount_rewards(rewards)
    episode_total_reward = rewards.shape[0]
    ep_total_lenghts.append(episode_total_reward)
    episodes += 1
    batch_actions += actions
    batch_disc_r = torch.concatenate([batch_disc_r, discounted_r])    
    batch_probs = torch.concatenate([batch_probs, probs])    

    if episodes % BATCH_SIZE == 0:
        loss = loss_function(batch_disc_r, batch_probs, batch_actions)
        if loss.isnan():
            break
        losses.append(loss.item())
        optim.zero_grad()
        loss.backward()
        optim.step()
        batch_actions = []
        batch_disc_r = torch.Tensor().to(DEVICE)
        batch_probs = torch.Tensor().to(DEVICE)
        print(f"Episode {episodes}. Loss: {loss:.3f}. Reward: {episode_total_reward}")
    
    if episodes % 50 == 0:
        torch.save(model.state_dict(), f"results/model_{episodes}_eps.pt")

torch.save(model.state_dict(), f"results/final_model_{episodes}_eps.pt")
    
print(f"Success in {episodes} episodes. Loss: {loss}. Reward: {episode_total_reward}")

In [ ]:
import plotly.express as px
px.line(losses, width=800, title="Loss over episodes")

In [ ]:
px.line(ep_total_lenghts, width=800, title="Rewards over episodes")

When the model trains, the entropy of its outputs decreases, as the agent is more certain on what action to take. At the beginning, the entropy is the maximum it can get with two actions (~0.69)

In [ ]:
px.line(entropies, width=800, title="Entropy over episodes")


## Adding entropy to the loss
Entropy: measure of uncertainty in a system. It shows how much the agent is uncertain about wich action to take. 

$H(\pi) = - \sum\pi(a|s)log\pi(a|s)$

It's always $ H(\pi)>0 $, and has a maximum when all actions have same probability. For example, we show here the maximum entropy for 2, 3 and 4 actions:

In [ ]:
import numpy as np
for n_actions in range(2, 5):
    prob_actions = np.tile(1/n_actions, n_actions)
    entropy = -(prob_actions * np.log(prob_actions)).sum()
    print(f"Maximum entropy with {n_actions} actions: {entropy}")


And entropy approaches zero as one of the actions has probability of 1 and the others, 0 probability:

In [ ]:
- (np.array([.999,0.001])*np.log([.999,0.001])).sum()

In [ ]:
ENTROPY_BETA = 0.01
def loss_function_entropy(discounted_r, probs, actions):
    logprobs = torch.log(probs)
    entropy = -(probs*probs.log()).sum(dim=1).mean()
    selected = logprobs[range(probs.shape[0]), actions]
    discounted_r = (discounted_r - discounted_r.mean()) / discounted_r.std() # This acts as removing the baseline. It's the simplest approach.
    weighted = selected * discounted_r
    loss = - weighted.mean() 
    # If the agent is very certain, the entropy will be low
    loss = loss - entropy * ENTROPY_BETA
    return loss

## Kullback-Leibler divergence
Kullback–Leibler (KL) divergence, is a type of statistical distance: a measure of how one reference probability distribution $P$ is different from a second probability distribution $Q$.

$ D_{KL} = \sum P(x)log(\frac{P(x)}{Q(x)})$

In our case, we can use $D_{KL}$ to see how different is the probability distribution of our policy before and after the weight updates. If there are big spikes in KL divergence, it means that we pushed the policy too much, and we don't want that. It's a way to monitor the change during training. High values can indicate a learning rate to high, for example. 

In [ ]:
def run_episode_returning_states(model, env, render=False):
    obs = env.reset()[0]
    if render:
        env.render()
    obs = torch.Tensor(env.reset()[0]).to(DEVICE)
    te = tr = False
    rewards, states, outputs, actions = [], [], [], []
    while not (te or tr):
        probs = model(obs)
        action = probs.multinomial(1).item()
        obs, r, te, tr, _ = env.step(action)
        obs = torch.Tensor(obs).to(DEVICE)
        r = -1 if te else 1 if tr else 0
        rewards.append(r)
        outputs.append(probs)
        actions.append(action)
        states.append(obs)
    return torch.Tensor(rewards).to(DEVICE), torch.concatenate(outputs).reshape(len(rewards), 2), actions, torch.concatenate(states).reshape(len(rewards), 4)

def kullback_leibler_divergence(model, old_probs, states):
    new_probs = model(states)
    kl_divergence = (old_probs * (new_probs / old_probs).log()).sum(dim=1).mean()
    return kl_divergence.detach().item()

In [ ]:
episode_total_reward = 0
batch_losses = torch.Tensor().to(DEVICE)
batch_actions = []
batch_disc_r = torch.Tensor().to(DEVICE)
batch_probs = torch.Tensor().to(DEVICE)
batch_states = torch.Tensor().to(DEVICE)
entropies_discounted = []
kl_divergences = []
best_ep_reward = 0
losses, ep_total_lenghts = [], [0]

episodes = 0
TARGET_REWARD = 295

env = gym.make("CartPole-v1")
model = VPG(env.observation_space.shape[0],
            2).to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
torch.save(model.state_dict(), "results/entropy_disc_model_0_eps.pt")

while np.mean(ep_total_lenghts[-100:]) < TARGET_REWARD:
    rewards, probs, actions, states = run_episode_returning_states(model, env)
    episode_entropy = -(probs * probs.log()).sum(dim=1).mean()
    entropies_discounted.append(episode_entropy.detach().item())
    discounted_r = discount_rewards(rewards)
    episode_total_reward = rewards.shape[0]
    ep_total_lenghts.append(episode_total_reward)
    episodes += 1
    batch_actions += actions
    batch_disc_r = torch.concatenate([batch_disc_r, discounted_r])    
    batch_probs = torch.concatenate([batch_probs, probs])    
    batch_states = torch.concatenate([batch_states, states])

    if episodes % BATCH_SIZE == 0:
        loss = loss_function_entropy(batch_disc_r, batch_probs, batch_actions)
        if loss.isnan():
            break
        losses.append(loss.item())
        optim.zero_grad()
        loss.backward()
        optim.step()
        kl_div = kullback_leibler_divergence(model, batch_probs, batch_states)
        kl_divergences.append(kl_div)
        batch_actions = []
        batch_disc_r = torch.Tensor().to(DEVICE)
        batch_probs = torch.Tensor().to(DEVICE)
        batch_states = torch.Tensor().to(DEVICE)
        print(f"Episode {episodes}. Loss: {loss:.3f}. Reward: {episode_total_reward}")
    
    if episodes % 100 == 0:
        torch.save(model.state_dict(), f"results/entropy_disc_model_{episodes}_eps.pt")

torch.save(model.state_dict(), f"results/final_model_entropy_disc_{episodes}_eps.pt")
    
print(f"Success in {episodes} episodes. Loss: {loss}. Reward: {episode_total_reward}")

In [ ]:
kl_divergences = [d.detach().item() for d in kl_divergences]

In [ ]:
import plotly.graph_objects as go
px.line(losses, width=800, title="Loss over episodes").show()
px.line(ep_total_lenghts, width=800, title="Returns over episodes").show()
px.line(kl_divergences, width=800, title="KL Divergence").show()
fig = go.Figure()
fig.add_traces([
    go.Scatter(y=entropies, x=list(range(len(entropies))), mode="lines", name="Entropy, not discounted"),
    go.Scatter(y=entropies_discounted, x=list(range(len(entropies_discounted))), mode="lines", name="Entropy, discounted"),
])
fig.update_layout(width=800)
# px.line(y=[entropies_discounted,entropies], width=800, title="Entropy, discounted
#  from the loss, over episodes").show()

When we discount the entropy from the loss, we punish the model for being too certain (too deterministic) when choosing its actions. Then, entropies remains relatively high.

# Karpathy's solution

In [ ]:
""" Trains an agent with (stochastic) Policy Gradients on Pong. Uses OpenAI Gym. """
import numpy as np
import pickle
import gymnasium as gym

# hyperparameters
H = 128 # number of hidden layer neurons
batch_size = 10 # every how many episodes to do a param update?
learning_rate = 1e-2
gamma = 0.99 # discount factor for reward
decay_rate = 0.99 # decay factor for RMSProp leaky sum of grad^2
resume = False # resume from previous checkpoint?
render = False

# model initialization
D = 4 # input dimensionality: 80x80 grid
if resume:
  model = pickle.load(open('save.p', 'rb'))
else:
  model = {}
  model['W1'] = np.random.randn(H,D) / np.sqrt(D) # "Xavier" initialization
  model['W2'] = np.random.randn(H) / np.sqrt(H)
  
grad_buffer = { k : np.zeros_like(v) for k,v in model.items() } # update buffers that add up gradients over a batch
rmsprop_cache = { k : np.zeros_like(v) for k,v in model.items() } # rmsprop memory

def sigmoid(x): 
  return 1.0 / (1.0 + np.exp(-x)) # sigmoid "squashing" function to interval [0,1]

# def prepro(I):
#   """ prepro 210x160x3 uint8 frame into 6400 (80x80) 1D float vector """
#   I = I[35:195] # crop
#   I = I[::2,::2,0] # downsample by factor of 2
#   I[I == 144] = 0 # erase background (background type 1)
#   I[I == 109] = 0 # erase background (background type 2)
#   I[I != 0] = 1 # everything else (paddles, ball) just set to 1
#   return I.astype(np.float).ravel()

def discount_rewards(r):
  """ take 1D float array of rewards and compute discounted reward """
  discounted_r = np.zeros_like(r)
  running_add = 0
  for t in reversed(range(r.size-1, -1, -1)):
    # if r[t] != 0: running_add = 0 # reset the sum, since this was a game boundary (pong specific!)
    running_add = running_add * gamma + r[t]
    discounted_r[t] = running_add
  return discounted_r

def policy_forward(x):
  h = np.dot(model['W1'], x)
  h[h<0] = 0 # ReLU nonlinearity
  logp = np.dot(model['W2'], h)
  p = sigmoid(logp)
  return p, h # return probability of taking action 2, and hidden state

def policy_backward(eph, epdlogp):
  """ backward pass. (eph is array of intermediate hidden states) """
  dW2 = np.dot(eph.T, epdlogp).ravel()
  dh = np.outer(epdlogp, model['W2'])
  dh[eph <= 0] = 0 # backpro prelu
  dW1 = np.dot(dh.T, epx)
  return {'W1':dW1, 'W2':dW2}

env = gym.make("CartPole-v1")
observation = env.reset()[0]
prev_x = None # used in computing the difference frame
xs,hs,dlogps,drs = [],[],[],[]
running_reward = None
reward_sum = 0
episode_number = 0
while True:
  if render: env.render()

  # preprocess the observation, set input to network to be difference image
  cur_x = observation
  # x = cur_x - prev_x if prev_x is not None else np.zeros(D)
  # prev_x = cur_x

  # forward the policy network and sample an action from the returned probability
  aprob, h = policy_forward(cur_x)
  try:
    action = 0 if np.random.uniform() < aprob else 1 # roll the dice!
  except:
    breakpoint()

  # record various intermediates (needed later for backprop)
  xs.append(x) # observation
  hs.append(h) # hidden state
  
  dlogps.append(action - aprob) # grad that encourages the action that was taken to be taken (see http://cs231n.github.io/neural-networks-2/#losses if confused)

  # step the environment and get new measurements
  observation, reward, te, tr, _ = env.step(action)
  reward_sum += reward

  drs.append(reward) # record reward (has to be done after we call step() to get reward for previous action)

  if (te or tr): # an episode finished
    episode_number += 1

    # stack together all inputs, hidden states, action gradients, and rewards for this episode
    epx = np.vstack(xs)
    eph = np.vstack(hs)
    epdlogp = np.vstack(dlogps)
    epr = np.vstack(drs)
    xs,hs,dlogps,drs = [],[],[],[] # reset array memory

    # compute the discounted reward backwards through time
    discounted_epr = discount_rewards(epr)
    # standardize the rewards to be unit normal (helps control the gradient estimator variance)
    discounted_epr -= np.mean(discounted_epr)
    discounted_epr /= np.std(discounted_epr)

    epdlogp *= discounted_epr # modulate the gradient with advantage (PG magic happens right here.)
    grad = policy_backward(eph, epdlogp)
    for k in model: grad_buffer[k] += grad[k] # accumulate grad over batch

    # perform rmsprop parameter update every batch_size episodes
    if episode_number % batch_size == 0:
      for k,v in model.items():
        g = grad_buffer[k] # gradient
        rmsprop_cache[k] = decay_rate * rmsprop_cache[k] + (1 - decay_rate) * g**2
        model[k] += learning_rate * g / (np.sqrt(rmsprop_cache[k]) + 1e-5)
        grad_buffer[k] = np.zeros_like(v) # reset batch gradient buffer

    # boring book-keeping
    running_reward = reward_sum if running_reward is None else running_reward * 0.99 + reward_sum * 0.01
    print ('resetting env. episode reward total was %f. running mean: %f' % (reward_sum, running_reward))
    if episode_number % 100 == 0: pickle.dump(model, open('save.p', 'wb'))
    reward_sum = 0
    observation = env.reset()[0] # reset env
    prev_x = None

In [ ]:
import numpy as np

probs =np.tile(1/6, 6)

(probs*np.log(probs)).sum()